In [13]:
import numpy as np
import pandas as pd 
import os
import requests
import json 
from bs4 import BeautifulSoup
import boto3
from IPython.display import HTML, display
import time 


In [14]:
url="https://rehos.cuni.cz/crpp/eshop/index"
headers={"User-Agent": "Anna_Chodurkova (Educational access; contact: chodurkovaanna@gmail.com)"}

In [15]:
r=requests.get(url,headers=headers)
r.text

'<!DOCTYPE html>\n<html>\n  <head>\n    <title>Centrální prodejní a rezervační portál ubytování na kolejích UK | Centrální prodejní a rezervační portál ubytování na kolejích UK</title>\n    <meta http-equiv="Content-Type" content="text/html; charset=UTF-8"/>\n    <meta name="Description" content="Centr&aacute;ln&iacute; prodejn&iacute; a rezervačn&iacute; port&aacute;l ubytov&aacute;n&iacute; na kolej&iacute;ch UK"/>\n    <link rel="stylesheet" href="/crpp/css/default.css" />\n    <link rel="stylesheet" href="/crpp/css/content.css" />\n    <link rel="stylesheet" href="/crpp/css/header.css" />\n    <link rel="stylesheet" href="/crpp/css/footer.css" />\n    <link rel="stylesheet" href="/crpp/css/main.css" />\n    <link rel="shortcut icon" href="/crpp/images/favicon.ico" type="image/x-icon" />\n    \n<!--<link rel="stylesheet" href="/crpp/css/jquery-ui-1.8.9.custom.css"/>-->\n<script src="/crpp/js/console-log-reset.js"></script>\n<link rel="shortcut icon" href="/crpp/images/favicon.ico" t

In [16]:
soup=BeautifulSoup(r.text,"html")

In [17]:
dorm_id=[380942,380952,380944,380939,380946,380958,380945,380950,380943,380948,380951,380961]

In [12]:
data=[]
for id in dorm_id:
    url=f"https://rehos.cuni.cz/crpp/eshop/collegeDetail/{id}"
    soup=BeautifulSoup(requests.get(url).content,"html.parser")
    #find dorm name
    dorm_name = soup.find("h1").text.strip()
    
    #find room data
    table = soup.find("table", {"class": "form"})
    #loop through each table row 
    for row in table.find_all("tr"):
        #find all cells
        cols = row.find_all("td")
        #avoid header rows
        if len(cols) >= 2:
            #data, clean
            data.append({
                "dorm": dorm_name,
                "room_type": cols[0].text.strip().split("\n")[0].lower(),
                "price": cols[1].text.strip().split("\n")[0]
            })
    time.sleep(1)






In [26]:
#further cleaning
df=pd.DataFrame(data)
df


,dorm,room_type,price
0,Kolej Budeč,1-lůžkový pokoj bez soc. zař.,230
1,Kolej Budeč,2-lůžkový pokoj bez soc. zař.,176
2,Kolej Budeč,3-lůžkový pokoj bez soc. zař.,146
3,Kolej Hostivař,1-lůžkový pokoj se soc. zař.,196
4,Kolej Hostivař,2-lůžkový pokoj bez soc. zař.,151
5,Kolej Hostivař,2-lůžkový pokoj se soc. zař.,164
6,Kolej Hvězda,1-lůžkový pokoj bez soc. zař.,198
7,Kolej Hvězda,2-lůžkový pokoj bez soc. zař.,155
8,Kolej Hvězda,2-lůžkový pokoj se soc. zař.,242
9,Kolej Hvězda,1-lůžkový pokoj se soc. zař.,269


In [29]:
df = df[df["price"].astype(int) != 0]

for index, row in df.iterrows():
    if 'nw' in row['room_type'] or 'nwxl' in row['room_type']:
        df = df.drop(index)

df = df.reset_index(drop=True)
df

,dorm,room_type,price
0,Kolej Budeč,1-lůžkový pokoj bez soc. zař.,230
1,Kolej Budeč,2-lůžkový pokoj bez soc. zař.,176
2,Kolej Budeč,3-lůžkový pokoj bez soc. zař.,146
3,Kolej Hostivař,1-lůžkový pokoj se soc. zař.,196
4,Kolej Hostivař,2-lůžkový pokoj bez soc. zař.,151
5,Kolej Hostivař,2-lůžkový pokoj se soc. zař.,164
6,Kolej Hvězda,1-lůžkový pokoj bez soc. zař.,198
7,Kolej Hvězda,2-lůžkový pokoj bez soc. zař.,155
8,Kolej Hvězda,2-lůžkový pokoj se soc. zař.,242
9,Kolej Hvězda,1-lůžkový pokoj se soc. zař.,269


In [30]:
df.to_csv("dorm_prices.csv",index=False)

In [31]:
df=pd.read_csv("dorm_prices.csv")
df

,dorm,room_type,price
0,Kolej Budeč,1-lůžkový pokoj bez soc. zař.,230
1,Kolej Budeč,2-lůžkový pokoj bez soc. zař.,176
2,Kolej Budeč,3-lůžkový pokoj bez soc. zař.,146
3,Kolej Hostivař,1-lůžkový pokoj se soc. zař.,196
4,Kolej Hostivař,2-lůžkový pokoj bez soc. zař.,151
5,Kolej Hostivař,2-lůžkový pokoj se soc. zař.,164
6,Kolej Hvězda,1-lůžkový pokoj bez soc. zař.,198
7,Kolej Hvězda,2-lůžkový pokoj bez soc. zař.,155
8,Kolej Hvězda,2-lůžkový pokoj se soc. zař.,242
9,Kolej Hvězda,1-lůžkový pokoj se soc. zař.,269
